# 法律名検索、条文内キーワード検索


In [6]:
import requests
import xml.etree.ElementTree as ET

# ============================================================
# ステップ①②：法律名をキーワードで検索する
# ============================================================
law_keyword = input("法律名のキーワードを入力してください（例：育児、労働、社会保険）：")

url = "https://laws.e-gov.go.jp/api/1/lawlists/2"
response = requests.get(url)
root_list = ET.fromstring(response.content)

# キーワードに合致する法律を抽出
matched_laws = []
for law in root_list.findall(".//LawNameListInfo"):
    law_name = law.find("LawName").text
    law_id   = law.find("LawId").text
    if law_keyword in law_name:
        matched_laws.append({"name": law_name, "id": law_id})

if len(matched_laws) == 0:
    print("該当する法律が見つかりませんでした。")
else:
    print(f"\n「{law_keyword}」を含む法律：{len(matched_laws)}件")
    print("=" * 50)
    for i, law in enumerate(matched_laws):
        print(f"{i+1}. {law['name']}")
    print("=" * 50)

    # ============================================================
    # ステップ③：法律を番号で選ぶ
    # ============================================================
    while True:
        try:
            choice = int(input(f"\n番号を選んでください（1〜{len(matched_laws)}）："))
            if 1 <= choice <= len(matched_laws):
                break
            else:
                print(f"1〜{len(matched_laws)}の番号を入力してください")
        except ValueError:
            print("数字を入力してください")

    selected = matched_laws[choice - 1]
    print(f"\n✅ 選択：{selected['name']}")

    # ============================================================
    # ステップ④：条文内をキーワードで検索する
    # ============================================================
    article_keyword = input("条文内の検索キーワードを入力してください（表示は本則→附則の順番）：")

    law_url = f"https://laws.e-gov.go.jp/api/1/lawdata/{selected['id']}"
    law_response = requests.get(law_url)
    law_root = ET.fromstring(law_response.content)

    # ============================================================
    # ステップ⑤：該当条文を表示する
    # ============================================================
    articles = law_root.findall(".//Article")
    count = 0

    print(f"\n「{article_keyword}」を含む条文：")
    print("=" * 50)

    for article in articles:
        num = article.get("Num", "")
        sentences = article.findall(".//Sentence")
        text = "".join([s.text for s in sentences if s.text])

        if article_keyword in text:
            print(f"\n第{num}条")
            print(f"{text[:200]}...")
            count += 1

    print(f"\n{'=' * 50}")
    if count == 0:
        print(f"「{article_keyword}」を含む条文は見つかりませんでした。")
    else:
        print(f"「{article_keyword}」を含む条文：{count}件")

法律名のキーワードを入力してください（例：育児、労働、社会保険）：労働

「労働」を含む法律：39件
1. 労働関係調整法
2. 労働者災害補償保険法
3. 労働基準法
4. 行政執行法人の労働関係に関する法律
5. 公共企業体労働関係法の施行に関する法律
6. 労働組合法
7. 駐留軍労働者等に支払うべき給料その他の給与の支払事務の処理の特例に関する法律
8. 地方公営企業等の労働関係に関する法律
9. 労働金庫法
10. 財団法人労働科学研究所に対する国有財産の譲与に関する法律
11. 労働保険審査官及び労働保険審査会法
12. 労働災害防止団体法
13. 労働施策の総合的な推進並びに労働者の雇用の安定及び職業生活の充実等に関する法律
14. 失業保険法及び労働者災害補償保険法の一部を改正する法律及び労働保険の保険料の徴収等に関する法律の施行に伴う関係法律の整備等に関する法律
15. 労働保険の保険料の徴収等に関する法律
16. 家内労働法
17. 労働安全衛生法
18. 建設労働者の雇用の改善等に関する法律
19. 労働者派遣事業の適正な運営の確保及び派遣労働者の保護等に関する法律
20. 港湾労働法
21. 中小企業における労働力の確保及び良好な雇用の機会の創出のための雇用管理の改善の促進に関する法律
22. 育児休業、介護休業等育児又は家族介護を行う労働者の福祉に関する法律
23. 介護労働者の雇用管理の改善等に関する法律
24. 労働時間等の設定の改善に関する特別措置法
25. 短時間労働者及び有期雇用労働者の雇用管理の改善等に関する法律
26. 林業労働力の確保の促進に関する法律
27. 厚生労働省設置法
28. 独立行政法人駐留軍等労働者労務管理機構法
29. 会社分割に伴う労働契約の承継等に関する法律
30. 個別労働関係紛争の解決の促進に関する法律
31. 独立行政法人労働者健康安全機構法
32. 独立行政法人労働政策研究・研修機構法
33. 労働審判法
34. 労働契約法
35. 専門的知識等を有する有期雇用労働者等に関する特別措置法
36. 労働者の職務に応じた待遇の確保等のための施策の推進に関する法律
37. 労働者協同組合法
38. 特定石綿被害建設業務労働者等に対する給付金等の支給に関する法律
39. 中小事業主が行う事業に従事する者等

# 練習用

In [2]:
import requests
import xml.etree.ElementTree as ET

# Version1の正しいエンドポイントで法令リストを取得
# 2=法律のみを取得
url = "https://laws.e-gov.go.jp/api/1/lawlists/2"

response = requests.get(url)
print(f"ステータスコード：{response.status_code}")

if response.status_code == 200:
    # XMLを解析する
    root = ET.fromstring(response.content)

    # 法令名を最初の10件だけ表示
    laws = root.findall(".//LawNameListInfo")
    print(f"取得した法律の数：{len(laws)}件")
    print("---最初の10件---")
    for law in laws[:10]:
        law_id   = law.find("LawId").text
        law_name = law.find("LawName").text
        print(f"{law_name}（ID：{law_id}）")

ステータスコード：200
取得した法律の数：2101件
---最初の10件---
日本国憲法（ID：321CONSTITUTION）
明治六年太政官布告第六十五号（絞罪器械図式）（ID：106DF0000000065）
明治十七年太政官布告第三十二号（爆発物取締罰則）（ID：117DF1000000032）
明治二十二年法律第三十四号（決闘罪ニ関スル件）（ID：122AC0000000034）
保管金規則（ID：123AC0000000001）
通貨及証券模造取締法（ID：128AC0000000028）
国債証券買入銷却法（ID：129AC0000000005）
民法（ID：129AC0000000089）
砂防法（ID：130AC0000000029）
民法施行法（ID：131AC0000000011）


In [3]:
# 取得した法令リストから「労働」を含む法律を検索する
lawnum = 0
print("「労働」を含む法律一覧：")
for law in laws:
    law_name = law.find("LawName").text
    law_id   = law.find("LawId").text
    if "労働" in law_name:
        print(f"  {law_name}（ID：{law_id}）")
        lawnum += 1
print(f"「労働」を含む法律の件数：{lawnum}件：")

「労働」を含む法律一覧：
  労働関係調整法（ID：321AC0000000025）
  労働者災害補償保険法（ID：322AC0000000050）
  労働基準法（ID：322AC0000000049）
  行政執行法人の労働関係に関する法律（ID：323AC0000000257）
  公共企業体労働関係法の施行に関する法律（ID：324AC0000000083）
  労働組合法（ID：324AC0000000174）
  駐留軍労働者等に支払うべき給料その他の給与の支払事務の処理の特例に関する法律（ID：325AC0000000005）
  地方公営企業等の労働関係に関する法律（ID：327AC0000000289）
  労働金庫法（ID：328AC0100000227）
  財団法人労働科学研究所に対する国有財産の譲与に関する法律（ID：328AC1000000224）
  労働保険審査官及び労働保険審査会法（ID：331AC0000000126）
  労働災害防止団体法（ID：339AC0000000118）
  労働施策の総合的な推進並びに労働者の雇用の安定及び職業生活の充実等に関する法律（ID：341AC0000000132）
  失業保険法及び労働者災害補償保険法の一部を改正する法律及び労働保険の保険料の徴収等に関する法律の施行に伴う関係法律の整備等に関する法律（ID：344AC0000000085）
  労働保険の保険料の徴収等に関する法律（ID：344AC0000000084）
  家内労働法（ID：345AC0000000060）
  労働安全衛生法（ID：347AC0000000057）
  建設労働者の雇用の改善等に関する法律（ID：351AC0000000033）
  労働者派遣事業の適正な運営の確保及び派遣労働者の保護等に関する法律（ID：360AC0000000088）
  港湾労働法（ID：363AC0000000040）
  中小企業における労働力の確保及び良好な雇用の機会の創出のための雇用管理の改善の促進に関する法律（ID：403AC0000000057）
  育児休業、介護休業等育児又は家族介護を行う労働者の福祉に関する法律（ID：403AC0000000076）
  介護労働者の雇用管理の改善等に関する法律（ID：404AC

In [4]:
# 労働基準法の条文を取得する
# IDは先ほどの結果から確認：322AC0000000049
law_id = "322AC0000000049"
url = f"https://laws.e-gov.go.jp/api/1/lawdata/{law_id}"

response = requests.get(url)
print(f"ステータスコード：{response.status_code}")

if response.status_code == 200:
    root = ET.fromstring(response.content)

    # 法令名を取得
    law_title = root.find(".//LawTitle")
    if law_title is not None:
        print(f"法令名：{law_title.text}")

    # 条文を最初の5件表示
    print("\n---条文（最初の5件）---")
    articles = root.findall(".//Article")
    for article in articles[:5]:
        # 条番号
        num = article.get("Num", "")
        # 条文のテキストを取得
        sentences = article.findall(".//Sentence")
        text = "".join([s.text for s in sentences if s.text])
        print(f"\n第{num}条")
        print(f"{text[:100]}...")  # 最初の100文字

ステータスコード：200
法令名：労働基準法

---条文（最初の5件）---

第1条
労働条件は、労働者が人たるに値する生活を営むための必要を充たすべきものでなければならない。この法律で定める労働条件の基準は最低のものであるから、労働関係の当事者は、この基準を理由として労働条件を低下さ...

第2条
労働条件は、労働者と使用者が、対等の立場において決定すべきものである。労働者及び使用者は、労働協約、就業規則及び労働契約を遵守し、誠実に各々その義務を履行しなければならない。...

第3条
使用者は、労働者の国籍、信条又は社会的身分を理由として、賃金、労働時間その他の労働条件について、差別的取扱をしてはならない。...

第4条
使用者は、労働者が女性であることを理由として、賃金について、男性と差別的取扱いをしてはならない。...

第5条
使用者は、暴行、脅迫、監禁その他精神又は身体の自由を不当に拘束する手段によつて、労働者の意思に反して労働を強制してはならない。...


In [5]:
# キーワードを入力して条文を検索するツール
keyword = input("検索キーワードを入力してください：")

print(f"\n「{keyword}」を含む条文：")
print("=" * 50)

articles = root.findall(".//Article")
count = 0

for article in articles:
    num = article.get("Num", "")
    sentences = article.findall(".//Sentence")
    text = "".join([s.text for s in sentences if s.text])

    if keyword in text:
        print(f"\n第{num}条")
        print(f"{text[:200]}...")
        count += 1

print(f"\n{'=' * 50}")
print(f"「{keyword}」を含む条文：{count}件")

検索キーワードを入力してください：時間外

「時間外」を含む条文：

第36条
使用者は、当該事業場に、労働者の過半数で組織する労働組合がある場合においてはその労働組合、労働者の過半数で組織する労働組合がない場合においては労働者の過半数を代表する者との書面による協定をし、厚生労働省令で定めるところによりこれを行政官庁に届け出た場合においては、第三十二条から第三十二条の五まで若しくは第四十条の労働時間（以下この条において「労働時間」という。）又は前条の休日（以下この条において「...

第37条
使用者が、第三十三条又は前条第一項の規定により労働時間を延長し、又は休日に労働させた場合においては、その時間又はその日の労働については、通常の労働時間又は労働日の賃金の計算額の二割五分以上五割以下の範囲内でそれぞれ政令で定める率以上の率で計算した割増賃金を支払わなければならない。ただし、当該延長して労働させた時間が一箇月について六十時間を超えた場合においては、その超えた時間の労働については、通常の...

第56条
使用者は、児童が満十五歳に達した日以後の最初の三月三十一日が終了するまで、これを使用してはならない。前項の規定にかかわらず、別表第一第一号から第五号までに掲げる事業以外の事業に係る職業で、児童の健康及び福祉に有害でなく、かつ、その労働が軽易なものについては、行政官庁の許可を受けて、満十三歳以上の児童をその者の修学時間外に使用することができる。映画の製作又は演劇の事業については、満十三歳に満たない児...

第66条
使用者は、妊産婦が請求した場合においては、第三十二条の二第一項、第三十二条の四第一項及び第三十二条の五第一項の規定にかかわらず、一週間について第三十二条第一項の労働時間、一日について同条第二項の労働時間を超えて労働させてはならない。使用者は、妊産婦が請求した場合においては、第三十三条第一項及び第三項並びに第三十六条第一項の規定にかかわらず、時間外労働をさせてはならず、又は休日に労働させてはならない...

第133条
厚生労働大臣は、第三十六条第二項の基準を定めるに当たつては、満十八歳以上の女性のうち雇用の分野における男女の均等な機会及び待遇の確保等のための労働省関係法律の整備に関する法律（平成九年法律第九十二号）第四条の規定による改正前の第六十四条の